# Training



In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path('/kaggle/working/PA2')
assert (REPO_ROOT / 'submission_adapter.py').is_file(), (
    ''
)
%cd /kaggle/working/PA2


In [ ]:
print('Using the packages already installed.')


In [ ]:
import gc
import json
import shutil
from pathlib import Path

import torch
import transformers
import datasets
import trl
import peft

GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('trl:', trl.__version__)
print('peft:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', GPU_COUNT)
print('GPU names:', GPU_NAMES)
!nvidia-smi
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
if GPU_COUNT < 2:
    print('WARNING: the assignment recommends 2 GPUs. The code can fall back to 1 GPU, but memory use may be higher.')

from data_utils import set_seed, load_sft_data, load_hh_preferences, load_ultrafeedback_data
from sft_training import train_sft
from dpo_training import train_dpo
from persona_lora_training import train_persona_dpo
from transformers import AutoModelForCausalLM, AutoTokenizer

set_seed(42)
BASE_MODEL = 'Qwen/Qwen2.5-0.5B'
OUTPUT_ROOT = REPO_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Supervised fine-tuning

Manual PyTorch SFT on 30,000 training and 1,000 validation examples. The training implementation applies response-only loss masking and saves best/final checkpoints plus epoch metrics.


In [ ]:
def manual_stage_complete(run_dir: Path) -> bool:
    return (run_dir / 'checkpoint-best').is_dir() and (run_dir / 'checkpoint-final').is_dir()

sft_dir = OUTPUT_ROOT / 'sft'
if manual_stage_complete(sft_dir):
    print('SFT is complete; reusing existing best checkpoint.')
else:
    if sft_dir.exists():
        print('Removing incomplete SFT output; this manual loop cannot restore optimizer state.')
        shutil.rmtree(sft_dir)
    sft_train, sft_valid = load_sft_data(train_size=30000, validation_size=1000, seed=42)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    sft_result = train_sft(
        model, tokenizer, sft_train, sft_valid, sft_dir,
        epochs=1, batch_size=2, grad_accum_steps=16, learning_rate=2e-5,
        weight_decay=0.1, warmup_ratio=0.03, max_length=512,
        max_grad_norm=1.0, seed=42, use_amp=True, device='cuda:0',
    )
    print(sft_result)
    del model, tokenizer, sft_train, sft_valid
    gc.collect()
    torch.cuda.empty_cache()

SFT_BEST = sft_dir / 'checkpoint-best'
assert SFT_BEST.is_dir() and (sft_dir / 'checkpoint-final').is_dir(), 'SFT did not produce both best and final checkpoints.'


## Manual DPO: HH-RLHF and UltraFeedback

Both runs initialize the policy and frozen reference from the SFT best checkpoint. With 2 GPUs, the reference is placed on `cuda:1`; otherwise it shares `cuda:0`. Each dataset uses 15,000 training pairs and 1,000 validation pairs.


In [ ]:
def run_manual_dpo(track_name, train_rows, valid_rows):
    run_dir = OUTPUT_ROOT / track_name
    if manual_stage_complete(run_dir):
        print(track_name, 'is complete; reusing existing best checkpoint.')
        return {'reused': True, 'checkpoint': str(run_dir / 'checkpoint-best')}
    if run_dir.exists():
        print('Removing incomplete', track_name, 'output; this manual loop cannot restore optimizer state.')
        shutil.rmtree(run_dir)

    tokenizer = AutoTokenizer.from_pretrained(SFT_BEST)
    policy = AutoModelForCausalLM.from_pretrained(SFT_BEST, torch_dtype=torch.float32)
    reference_dtype = torch.float16 if GPU_COUNT > 1 else torch.float32
    reference = AutoModelForCausalLM.from_pretrained(SFT_BEST, torch_dtype=reference_dtype)
    reference_device = 'cuda:1' if GPU_COUNT > 1 else 'cuda:0'
    result = train_dpo(
        policy, reference, tokenizer, train_rows, valid_rows, run_dir,
        beta=0.1, epochs=1, batch_size=1, grad_accum_steps=32,
        learning_rate=1e-6, max_length=512, max_grad_norm=1.0, seed=42,
        use_amp=True, policy_device='cuda:0', reference_device=reference_device,
        reference_fp16=(GPU_COUNT > 1),
    )
    del policy, reference, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    assert manual_stage_complete(run_dir), f'{track_name} did not produce both best and final checkpoints.'
    return result

hh_dir = OUTPUT_ROOT / 'dpo_hh'
if manual_stage_complete(hh_dir):
    print('HH-RLHF DPO is complete; reusing existing checkpoint.')
else:
    hh_train, hh_valid = load_hh_preferences(train_size=15000, validation_size=1000, seed=42)
    print('HH-RLHF:', run_manual_dpo('dpo_hh', hh_train, hh_valid))
    del hh_train, hh_valid
    gc.collect()
    torch.cuda.empty_cache()

uf_dir = OUTPUT_ROOT / 'dpo_ultrafeedback'
if manual_stage_complete(uf_dir):
    print('UltraFeedback DPO is complete; reusing existing checkpoint.')
else:
    uf_train, uf_valid = load_ultrafeedback_data(train_size=15000, validation_size=1000, seed=42)
    print('UltraFeedback:', run_manual_dpo('dpo_ultrafeedback', uf_train, uf_valid))
    del uf_train, uf_valid
    gc.collect()
    torch.cuda.empty_cache()


## Persona DPO with TRL and LoRA

This separate experiment uses `HuggingFaceTB/SmolLM2-360M-Instruct` and the provided preference JSON. The training function handles TRL API differences and can resume from a saved TRL checkpoint.


In [ ]:
persona_dir = OUTPUT_ROOT / 'persona_lora'
persona_result_path = persona_dir / 'result.json'
if (persona_dir / 'adapter_config.json').is_file() and persona_result_path.is_file():
    print('Persona LoRA DPO is complete; reusing the existing adapter.')
    persona_result = json.loads(persona_result_path.read_text(encoding='utf-8'))
else:
    persona_result = train_persona_dpo(
        output_dir=persona_dir,
        data_path=REPO_ROOT / 'data/persona_lora/ramsay_style_preferences.json',
        seed=42, beta=0.1, learning_rate=5e-5,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=3, max_length=512, max_prompt_length=256,
        lora_r=8, lora_alpha=16, lora_dropout=0.05, resume=True,
    )
    print(persona_result)
assert (persona_dir / 'adapter_config.json').is_file(), 'Persona LoRA adapter was not saved.'


## Metrics and archive

Plots are created only for metric files that exist. The archive contains the checkpoints, configurations, metrics and plots under `outputs/`. Save a Kaggle notebook version after this cell completes.


In [ ]:
import matplotlib.pyplot as plt

def read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

plot_dir = OUTPUT_ROOT / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

sft_metrics = read_jsonl(OUTPUT_ROOT / 'sft' / 'metrics.jsonl')
if sft_metrics:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([m['epoch'] for m in sft_metrics], [m['train_loss'] for m in sft_metrics], marker='o', label='Train')
    ax.plot([m['epoch'] for m in sft_metrics], [m['validation_loss'] for m in sft_metrics], marker='o', label='Validation')
    ax.set(xlabel='Epoch', ylabel='Response-token loss', title='SFT loss')
    ax.legend()
    fig.tight_layout()
    fig.savefig(plot_dir / 'sft_loss.png', dpi=160)
    plt.show()
    plt.close(fig)

for run_name in ('dpo_hh', 'dpo_ultrafeedback'):
    metrics = read_jsonl(OUTPUT_ROOT / run_name / 'metrics.jsonl')
    if not metrics:
        print('No metrics found for', run_name)
        continue
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot([m['epoch'] for m in metrics], [m['validation_dpo_loss'] for m in metrics], marker='o')
    axes[0].set(xlabel='Epoch', ylabel='DPO loss', title=f'{run_name}: validation loss')
    axes[1].plot([m['epoch'] for m in metrics], [m['validation_reward_accuracy'] for m in metrics], marker='o', label='Reward accuracy')
    axes[1].plot([m['epoch'] for m in metrics], [m['validation_reward_margin'] for m in metrics], marker='o', label='Reward margin')
    axes[1].set(xlabel='Epoch', title=f'{run_name}: preference metrics')
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(plot_dir / f'{run_name}_metrics.png', dpi=160)
    plt.show()
    plt.close(fig)

manifest = {
    'base_model': BASE_MODEL, 'seed': 42, 'gpu_count': GPU_COUNT, 'gpu_names': GPU_NAMES,
    'torch': torch.__version__, 'transformers': transformers.__version__,
    'datasets': datasets.__version__, 'trl': trl.__version__, 'peft': peft.__version__,
}
(OUTPUT_ROOT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/training_outputs', 'zip', root_dir=OUTPUT_ROOT)
print('Training artifacts:', OUTPUT_ROOT)
print('Archive:', archive)
